In [ ]:
#new attempt at CNN on this dataset
#Alex Stedman
#we will be running a CNN on these spectrogram images to classify them
#use torchvision
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim

#specify training and validation directories
train_dir = "../Datasets/SmellySongs9k/train"
val_dir = "../Datasets/SmellySongs9k/test"

#read out the dimensions of the images
example_image = plt.imread(os.path.join(train_dir, os.listdir(train_dir)[0], os.listdir(os.path.join(train_dir, os.listdir(train_dir)[0]))[0]))
img_height, img_width = example_image.shape[:2]
img_channels = example_image.shape[2] if len(example_image.shape) == 3 else 1
print(f"Image dimensions: {img_height}x{img_width}")
print(f"Image channels: {img_channels}")

if(img_height != 224 or img_width != 224):
    print("Warning: Images are not 224x224 pixels. SimpleCNN1 expects 224x224 pixel images! Please see SimpleCNN1.py")


In [ ]:
#setup the datasets and dataloaders
batch_size = 32
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((img_height, img_width)),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((img_height, img_width)),
        transforms.ToTensor(),
    ]),
}   
image_datasets = {
    'train': datasets.ImageFolder(train_dir, transform=data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, transform=data_transforms['val']),
}
dataloaders = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
    'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False),
}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

#print out data stats
for phase in ['train', 'val']:
    print(f"{phase} dataset size: {dataset_sizes[phase]} images")


In [ ]:
#OPTIONAL SHRINK THE TRAINING AND VALIDATION DATASETS FOR TESTING. 
shrink_data = False
if shrink_data:
    small_train_size = 300
    small_val_size = 300
    image_datasets['train'], _ = torch.utils.data.random_split(image_datasets['train'], [small_train_size, dataset_sizes['train'] - small_train_size])
    image_datasets['val'], _ = torch.utils.data.random_split(image_datasets['val'], [small_val_size, dataset_sizes['val'] - small_val_size])
    dataloaders['train'] = torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True)
    dataloaders['val'] = torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    dataset_sizes['train'] = small_train_size
    dataset_sizes['val'] = small_val_size
    print(f"Shrunk training dataset size: {dataset_sizes['train']} images")
    print(f"Shrunk validation dataset size: {dataset_sizes['val']} images")



In [ ]:
#print out the class split in the datasets
print("Class distribution in training dataset:")
train_class_counts = np.zeros(num_classes, dtype=int)
for _, labels in dataloaders['train']:
    for label in labels:
        train_class_counts[label] += 1
for i, count in enumerate(train_class_counts):
    print(f"  {class_names[i]}: {count} images")
print("Class distribution in validation dataset:")
val_class_counts = np.zeros(num_classes, dtype=int)
for _, labels in dataloaders['val']:
    for label in labels:
        val_class_counts[label] += 1
for i, count in enumerate(val_class_counts):
    print(f"  {class_names[i]}: {count} images")


In [ ]:

from SimpleCNN1 import SimpleCNN

In [ ]:
#initialize the model, loss function, and optimizer
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
#am I using the gpu?
print(f"Using device: {device}")
model = SimpleCNN(num_classes=num_classes, img_height=img_height, img_width=img_width)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
#perform training
num_epochs = 20
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # Each epoch has a training and validation phase
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
        else:
            model.eval()   # Set model to evaluate mode

        running_loss = 0.0
        running_corrects = 0

        # Iterate over data.
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # backward + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

In [ ]:
# save the model as a new filenames
n = 1
while os.path.exists(f"SimpleCNN1_T{n}.pth"):
    n += 1
torch.save(model.state_dict(), f"SimpleCNN1_T{n}.pth")